# module-base-class-custom — faded example 3: Fill the recursive train propagation

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-base-class-custom`. Running the beacon reports progress on the `Backprop: Module base class custom` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Module base class custom` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-base-class-custom`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-base-class-custom"
DD_SUBTOPIC = "Backprop: Module base class custom"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

`train(mode)` sets the root's flag then propagates to children by calling `m.train(mode)` on each submodule, and returns `self` for chaining. Without the recursion, nested submodules would keep their old training state.

## Faded exercise 3

Complete `Module.train(mode)`. Setting the root flag and the return are given; fill in the propagation to submodules.

**Fill in:** the loop calling train(mode) on every submodule

In [ ]:
import numpy as np

class Module:
    def __init__(self):
        object.__setattr__(self, '_parameters', {})
        object.__setattr__(self, '_modules', {})
        object.__setattr__(self, 'training', True)
    def __setattr__(self, name, value):
        if isinstance(value, Module):
            self._modules[name] = value
        object.__setattr__(self, name, value)
    def train(self, mode=True):
        self.training = mode
        raise NotImplementedError()  # TODO: the loop calling train(mode) on every submodule
        return self
    def eval(self):
        return self.train(False)


def _test():
    class Inner(Module):
        pass
    class Outer(Module):
        def __init__(self):
            super().__init__()
            self.inner = Inner()
    net = Outer()
    assert net.training and net.inner.training
    ret = net.eval()
    # propagation: both root and child flipped
    assert net.training is False
    assert net.inner.training is False
    # chaining: returns self
    assert ret is net
    net.train()
    assert net.training is True and net.inner.training is True


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np

class Module:
    def __init__(self):
        object.__setattr__(self, '_parameters', {})
        object.__setattr__(self, '_modules', {})
        object.__setattr__(self, 'training', True)
    def __setattr__(self, name, value):
        if isinstance(value, Module):
            self._modules[name] = value
        object.__setattr__(self, name, value)
    def train(self, mode=True):
        self.training = mode
        for m in self._modules.values():
            m.train(mode)
        return self
    def eval(self):
        return self.train(False)
```
</details>